In [1]:
!pip install -q uv && uv pip install -q autogluon.tabular

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.9/24.9 MB 49.0 MB/s eta 0:00:00


In [2]:
import warnings
import numpy as np
import pandas as pd
from autogluon.tabular import TabularPredictor
from sklearn.preprocessing import KBinsDiscretizer
from sklearn.utils.class_weight import compute_class_weight

FOLDS = 5
ID = 'id'
TARGET = 'class'
warnings.filterwarnings('ignore')

# Data Loading & Original Dataset Preparation

In [3]:
train = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/train.csv")
test = pd.read_csv("/kaggle/input/competitions/playground-series-s6e6/test.csv")
original = pd.read_csv("/kaggle/input/datasets/fedesoriano/stellar-classification-dataset-sdss17/star_classification.csv") 

train_id = train[ID]
test_id = test[ID]

def get_spectral_type(g, r):
    return pd.cut(
        r - g, 
        [-np.inf, -1, -0.5, 0, np.inf],
        labels=['M', 'G/K', 'A/F', 'O/B']
    ).astype(str)

def get_galaxy_population(u, r):
    return pd.cut(
        u - r, 
        [-np.inf, 1.4, 2.2, np.inf],
        labels=['Blue_Cloud', 'Green_Valley', 'Red_Sequence']
    ).astype(str)

original['spectral_type'] = get_spectral_type(original['g'], original['r'])
original['galaxy_population'] = get_galaxy_population(original['u'], original['r'])

target_map = {'GALAXY': 'GALAXY', 'QSO': 'QSO', 'STAR': 'STAR'}
original[TARGET] = original[TARGET].map(target_map)
train[TARGET] = train[TARGET].map(target_map)

base_features = [col for col in train.columns if col not in [TARGET, ID]]
cat_cols_init = train.drop(columns=[ID, TARGET]).select_dtypes(include=['object']).columns.tolist()
num_cols_init = [c for c in train.drop(columns=[ID, TARGET]).select_dtypes(exclude=['object']).columns]

original_numeric_target = original[TARGET].map({'GALAXY': 0, 'QSO': 1, 'STAR': 2})
original_global_mean = original_numeric_target.mean()
original_global_median = original_numeric_target.median()

original_stats = {}
for col in base_features:
    if col in original.columns:
        orig_temp = original[[col]].copy()
        orig_temp[TARGET] = original_numeric_target
        if col in num_cols_init:
            orig_temp[col] = np.floor(orig_temp[col])
            
        stats = orig_temp.groupby(col)[TARGET].agg(['mean', 'median', 'std', 'skew', 'count']).reset_index()
        stats.columns = [col] + [f"orig_{col}_{s}" for s in ['mean', 'median', 'std', 'skew', 'count']]
        original_stats[col] = stats

# Feature Engineering Pipeline Definition

In [4]:
color_pairs = [('u', 'g'), ('g', 'r'), ('r', 'i'), ('i', 'z'), ('u', 'z')]
important_combos = sorted([
    ('alpha_cat_', 'delta_cat_'),
    ('u_cat_', 'z_cat_'),
    ('g_cat_', 'r_cat_'), 
])

category_map = {}

def feature_engineering(df, fit=False):
    df = df.copy()
    eps = 1e-6

    for col, stats_df in original_stats.items():
        if col in df.columns:
            df_key = df[[col]].copy()
            if col in num_cols_init:
                df_key[col] = np.floor(df_key[col])
                
            merged_stats = df_key.merge(stats_df, on=col, how='left').drop(columns=[col])
            df = pd.concat([df, merged_stats], axis=1)
            
            fill_values = {
                f"orig_{col}_mean": original_global_mean, f"orig_{col}_median": original_global_median,
                f"orig_{col}_std": 0.0, f"orig_{col}_skew": 0.0, f"orig_{col}_count": 0.0
            }
            df.fillna(value=fill_values, inplace=True)
            for s in ['mean', 'median', 'std', 'skew', 'count']:
                df[f"orig_{col}_{s}"] = df[f"orig_{col}_{s}"].astype('float32')

    if 'redshift' in df.columns:
        df['_is_star_redshift'] = (df['redshift'] <= 0).astype('int32')
        df['_dist_modulus_proxy'] = (5 * np.log10(np.abs(df['redshift']) + eps)).astype('float32')
        if 'r' in df.columns: df['_absolute_r_proxy'] = (df['r'] - df['_dist_modulus_proxy']).astype('float32')
        if 'g' in df.columns: df['_absolute_g_proxy'] = (df['g'] - df['_dist_modulus_proxy']).astype('float32')

    bands = ['u', 'g', 'r', 'i', 'z']
    if all(b in df.columns for b in bands):
        df['_ugriz_skew'] = df[bands].skew(axis=1).astype('float32')
        df['_ugriz_kurt'] = df[bands].kurt(axis=1).astype('float32')
        df['_ugriz_mean'] = df[bands].mean(axis=1).astype('float32')
        df['_ugriz_std'] = df[bands].std(axis=1).astype('float32')
        df['_ugriz_max_diff'] = (df[bands].max(axis=1) - df[bands].min(axis=1)).astype('float32')

    if 'alpha' in df.columns and 'delta' in df.columns:
        alpha_rad, delta_rad = np.radians(df['alpha']), np.radians(df['delta'])
        
        df['_coord_x'] = (np.cos(delta_rad) * np.cos(alpha_rad)).astype('float32')
        df['_coord_y'] = (np.cos(delta_rad) * np.sin(alpha_rad)).astype('float32')
        df['_coord_z'] = (np.sin(delta_rad)).astype('float32')
        
        ra_gp = np.radians(192.85948)
        dec_gp = np.radians(27.12825)
        l_cp = np.radians(122.93314)
        
        sin_b = np.sin(delta_rad) * np.sin(dec_gp) + np.cos(delta_rad) * np.cos(dec_gp) * np.cos(alpha_rad - ra_gp)
        df['_galactic_b'] = np.degrees(np.arcsin(np.clip(sin_b, -1.0, 1.0))).astype('float32')
        
        y_l = np.cos(delta_rad) * np.sin(alpha_rad - ra_gp)
        x_l = np.sin(delta_rad) * np.cos(dec_gp) - np.cos(delta_rad) * np.sin(dec_gp) * np.cos(alpha_rad - ra_gp)
        df['_galactic_l'] = np.degrees(l_cp - np.arctan2(y_l, x_l)).astype('float32') % 360.0
        
        df['_sin_alpha'] = np.sin(alpha_rad).astype('float32')
        df['_cos_alpha'] = np.cos(alpha_rad).astype('float32')
        df['_sin_delta'] = np.sin(delta_rad).astype('float32')
        df['_cos_delta'] = np.cos(delta_rad).astype('float32')
        
        df['_alpha_plus_delta'] = (df['alpha'] + df['delta']).astype('float32')
        df['_alpha_minus_delta'] = (df['alpha'] - df['delta']).astype('float32')
        df['_alpha_mod1'] = (df['alpha'] % 1).astype('float32')
        df['_delta_mod1'] = (df['delta'] % 1).astype('float32')
        df['_total_spatial_distance'] = np.sqrt(df['alpha']**2 + df['delta']**2).astype('float32')
    
    if 'redshift' in df.columns:
        df['_log_redshift'] = np.log1p(np.abs(df['redshift'])).astype('float32')
        df['_r_x_redshift'] = (df['r'] * df['redshift']).astype('float32')
        df['_g_/_redshift'] = (df['g'] / (df['redshift'] + eps)).astype('float32')
        df['_i_/_redshift'] = (df['i'] / (df['redshift'] + eps)).astype('float32')
        
    for a, b in color_pairs:
        if a in df.columns and b in df.columns:
            df[f"_{a}-{b}"] = (df[a] - df[b]).astype('float32')

    if all(b in df.columns for b in ['g', 'r', 'i']):
        df['_balmer_break_proxy'] = ((df['g'] - df['r']) - (df['r'] - df['i'])).astype('float32')

    if all(b in df.columns for b in ['u', 'g', 'r', 'i', 'z']):
        df['g_r'] = df['g'] - df['r']
        df['r_i'] = df['r'] - df['i']
        df['u_g'] = df['u'] - df['g']
        df['i_z'] = df['i'] - df['z']
        
        df['_color_cross_ug_gr'] = (df['u_g'] * df['g_r']).astype('float32')
        df['_color_cross_gr_ri'] = (df['g_r'] * df['r_i']).astype('float32')
        
        df['stellar_locus_dist'] = np.sqrt((df['g_r'] - 0.52)**2 + (df['r_i'] - 0.25)**2).astype('float32')
        df['qso_locus_dist'] = np.sqrt((df['g_r'] - 0.24)**2 + (df['r_i'] - 0.15)**2).astype('float32')
        df['galaxy_locus_dist'] = np.sqrt((df['u_g'] - 1.50)**2 + (df['g_r'] - 0.70)**2).astype('float32')

    for col in cat_cols_init:
        if fit:
            df[col] = pd.Categorical(df[col])
            category_map[col] = df[col].dtype.categories
        else:
            df[col] = pd.Categorical(df[col], categories=category_map[col])

    for col in num_cols_init:
        cat_name = f"{col}_cat_"
        floored_values = np.floor(df[col])
        if fit:
            df[cat_name] = pd.Categorical(floored_values)
            category_map[col] = df[cat_name].dtype.categories
        else:
            df[cat_name] = pd.Categorical(floored_values, categories=category_map[col])

    for col, bins_list in {'delta': [100, 500]}.items():
        for n_bins in bins_list:
            bin_name = f"{col}_{n_bins}_quantile_bin_"
            if fit:
                kb = KBinsDiscretizer(n_bins=n_bins, encode='ordinal', strategy='quantile', subsample=None)
                binned = kb.fit_transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = pd.Categorical(binned)
                category_map[bin_name] = kb
            else:
                binned = category_map[bin_name].transform(df[[col]]).ravel().astype('int32')
                df[bin_name] = pd.Categorical(binned)

    for cols in important_combos:
        combo_name = '_'.join(cols) + '_'
        combo_series = df[cols[0]].astype(str)
        for col in cols[1:]: 
            combo_series = combo_series + '_' + df[col].astype(str)
        
        if fit:
            df[combo_name] = pd.Categorical(combo_series)
            category_map[combo_name] = df[combo_name].dtype.categories
        else:
            df[combo_name] = pd.Categorical(combo_series, categories=category_map[combo_name])

    return df

# Feature Engineering Execution 

In [5]:
X_train = train.drop(columns=[ID, TARGET])
y_train = train[TARGET]
X_test = test.drop(columns=[ID])

X_train = feature_engineering(X_train, fit=True)
X_test = feature_engineering(X_test, fit=False)

SKEW_THRESHOLD = 1.5
UNIQUENESS_THRESHOLD = 15
current_num_cols = X_train.select_dtypes(exclude=['category', 'object']).columns.tolist()

for col in current_num_cols:
    if X_train[col].nunique() <= UNIQUENESS_THRESHOLD: continue
    if abs(X_train[col].skew()) > SKEW_THRESHOLD:
        X_train[col] = (np.sign(X_train[col]) * np.log1p(np.abs(X_train[col]))).astype('float32')
        X_test[col] = (np.sign(X_test[col]) * np.log1p(np.abs(X_test[col]))).astype('float32')

for col in X_train.columns.tolist():
    if isinstance(X_train[col].dtype, pd.CategoricalDtype) or X_train[col].dtype == 'object' or X_train[col].nunique() <= UNIQUENESS_THRESHOLD:
        X_train[col], X_test[col] = pd.Categorical(X_train[col]), pd.Categorical(X_test[col])

constant_cols = [col for col in X_train.columns if X_train[col].nunique() <= 1]
if constant_cols:
    X_train.drop(columns=constant_cols, inplace=True)
    X_test.drop(columns=constant_cols, inplace=True)

# AutoGluon Training

In [6]:
classes = np.unique(y_train)
class_weights = compute_class_weight(
    class_weight="balanced",
    classes=classes,
    y=y_train
)
weight_map = dict(zip(classes, class_weights))

X_train['_sample_weight'] = y_train.map(weight_map).astype('float32')

X_train[TARGET] = y_train

predictor = TabularPredictor(
    label=TARGET, 
    eval_metric='balanced_accuracy', 
    sample_weight='_sample_weight'
).fit(
    train_data=X_train,
    num_bag_folds=FOLDS,          
    num_bag_sets=1,
    auto_stack=True,              
    time_limit=3600*10,              
    presets='best_quality',
    ag_args_fit={"num_gpus": 2},
)

No path specified. Models will be saved in: "AutogluonModels/ag-20260607_084411"
Verbosity: 2 (Standard Logging)
=================== System Info ===================
AutoGluon Version:  1.5.0
Python Version:     3.12.13
Operating System:   Linux
Platform Machine:   x86_64
Platform Version:   #1 SMP Thu Apr 30 18:17:14 UTC 2026
CPU Count:          4
Pytorch Version:    2.10.0+cu128
CUDA Version:       12.8
GPU Memory:         GPU 0: 14.56/14.56 GB | GPU 1: 14.56/14.56 GB
Total GPU Memory:   Free: 29.13 GB, Allocated: 0.00 GB, Total: 29.13 GB
GPU Count:          2
Memory Avail:       29.29 GB / 31.35 GB (93.4%)
Disk Space Avail:   19.50 GB / 19.52 GB (99.9%)
Presets specified: ['best_quality']
Using hyperparameters preset: hyperparameters='zeroshot'
Setting dynamic_stacking from 'auto' to True. Reason: Enable dynamic_stacking when use_bag_holdout is disabled. (use_bag_holdout=False)
Stack configuration (auto_stack=True): num_stack_levels=1, num_bag_folds=5, num_bag_sets=1
DyStack is enabl

[1000]	valid_set's multi_logloss: 0.113066	valid_set's balanced_accuracy: 0.962476


[LightGBM] [Fatal] bin size 7674 cannot run on GPU
[LightGBM] [Fatal] bin size 7695 cannot run on GPU
[LightGBM] [Fatal] bin size 7709 cannot run on GPU
[LightGBM] [Fatal] bin size 7629 cannot run on GPU


[1000]	valid_set's multi_logloss: 0.108583	valid_set's balanced_accuracy: 0.963267


	0.964	 = Validation score   (balanced_accuracy)
	1290.96s	 = Training   runtime
	117.22s	 = Validation runtime
Fitting model: NeuralNetFastAI_r191_BAG_L1 ... Training model for up to 10297.67s of the 19252.57s of remaining time.
	Fitting 5 child models (S1F1 - S1F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=2)
Metric balanced_accuracy is not supported by this model - using log_loss instead
No improvement since epoch 7: early stopping
Metric balanced_accuracy is not supported by this model - using log_loss instead
Metric balanced_accuracy is not supported by this model - using log_loss instead
Metric balanced_accuracy is not supported by this model - using log_loss instead
No improvement since epoch 6: early stopping
Metric balanced_accuracy is not supported by this model - using log_loss instead
	0.9559	 = Validation score   (balanced_accuracy)
	4376.25s	 = Training   runtime
	9.63s	 = Validation runtime
Fitting model: CatBoost_r9_BAG_L1 ... Training 

...

[LightGBM] [Fatal] bin size 7709 cannot run on GPU


[1000]	valid_set's multi_logloss: 0.100414	valid_set's balanced_accuracy: 0.963898
[2000]	valid_set's multi_logloss: 0.0971835	valid_set's balanced_accuracy: 0.965163
[3000]	valid_set's multi_logloss: 0.097871	valid_set's balanced_accuracy: 0.965231


[LightGBM] [Fatal] bin size 7629 cannot run on GPU


[1000]	valid_set's multi_logloss: 0.103347	valid_set's balanced_accuracy: 0.963961
[2000]	valid_set's multi_logloss: 0.100415	valid_set's balanced_accuracy: 0.965035


	0.9654	 = Validation score   (balanced_accuracy)
	2695.76s	 = Training   runtime
	259.19s	 = Validation runtime
Fitting model: NeuralNetTorch_r22_BAG_L1 ... Training model for up to 2766.79s of the 11721.69s of remaining time.
	Fitting 5 child models (S1F1 - S1F5) | Fitting with SequentialLocalFoldFittingStrategy (sequential: cpus=2, gpus=2)
		module 'numpy' has no attribute 'in1d'
Detailed Traceback:
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/autogluon/tabular/trainer/abstract_trainer.py", line 2201, in _train_and_save
    model = self._train_single(**model_fit_kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/autogluon/tabular/trainer/abstract_trainer.py", line 2085, in _train_single
    model = model.fit(X=X, y=y, X_val=X_val, y_val=y_val, X_test=X_test, y_test=y_test, total_resources=total_resources, **model_fit_kwargs)
            ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^

# Leaderboard 

In [7]:
predictor.leaderboard(silent=True).style.background_gradient(subset=['score_val'], cmap='RdYlGn') 

,model,score_val,eval_metric,pred_time_val,fit_time,pred_time_val_marginal,fit_time_marginal,stack_level,can_infer,fit_order
0,WeightedEnsemble_L3,0.969014,balanced_accuracy,796.443805,17189.420801,0.144318,130.401214,3,True,36
1,CatBoost_BAG_L2,0.968985,balanced_accuracy,793.311983,16857.093602,1.940279,100.472806,2,True,27
2,CatBoost_r177_BAG_L2,0.968957,balanced_accuracy,793.080473,16844.276887,1.708770,87.656090,2,True,31
3,CatBoost_r9_BAG_L2,0.968753,balanced_accuracy,793.245609,16820.922175,1.873905,64.301379,2,True,34
4,LightGBM_r131_BAG_L2,0.968670,balanced_accuracy,792.650439,16870.890692,1.278735,114.269895,2,True,32
5,LightGBMXT_BAG_L2,0.968537,balanced_accuracy,793.518065,16929.989629,2.146361,173.368832,2,True,23
6,LightGBMLarge_BAG_L2,0.968303,balanced_accuracy,792.392810,16873.421649,1.021106,116.800852,2,True,30
7,LightGBM_BAG_L2,0.968271,balanced_accuracy,792.262975,16857.389879,0.891271,100.769083,2,True,24
8,WeightedEnsemble_L2,0.968116,balanced_accuracy,452.499055,9596.253298,0.168912,76.833380,2,True,21
9,LightGBM_r96_BAG_L2,0.967839,balanced_accuracy,795.039305,16953.796583,3.667601,197.175786,2,True,35


# Output File Generation

In [8]:
X_train_predict = X_train.drop(columns=[TARGET, '_sample_weight'])

oof_preds_prob = predictor.predict_proba(X_train_predict)
oof_df = pd.DataFrame({ID: train_id})
for cls in ['GALAXY', 'QSO', 'STAR']:
    oof_df[cls] = oof_preds_prob[cls]
oof_df.to_csv('oof_preds.csv', index=False)

test_preds_prob = predictor.predict_proba(X_test)
test_df = pd.DataFrame({ID: test_id})
for cls in ['GALAXY', 'QSO', 'STAR']:
    test_df[cls] = test_preds_prob[cls]
test_df.to_csv('test_preds.csv', index=False)

sub_preds = predictor.predict(X_test)
sub = pd.DataFrame({ID: test_id, TARGET: sub_preds})
sub.to_csv('submission.csv', index=False)
sub.head() 

,id,class
0,577347,GALAXY
1,577348,GALAXY
2,577349,GALAXY
3,577350,STAR
4,577351,GALAXY
